# Lecture: Self-Supervised Colorization

In the previous notebook we trained an autoencoder to compress and reconstruct images. The model required no labels — the input itself served as the training target. This idea is central to **self-supervised learning**: designing a pretext task that generates supervision from the data itself.

**Colorization** is a classic self-supervised pretext task:
- Convert an RGB image to the **Lab color space**
- Feed only the **L channel** (lightness, i.e. a grayscale version) to the network
- Predict the **ab channels** (color information)
- Supervise with MSE against the ground-truth ab channels — no class labels needed

The network must learn to understand the *content* of the image to hallucinate plausible colors, making colorization a proxy task for general visual representation learning.

We use **CIFAR-10** (32×32 RGB, 10 classes, 60,000 images) — small enough to train in ~10 minutes on a Colab GPU while still containing meaningful color structure.

Run the following cell only if you are working with Google Colab to copy the required .py file into the root directory. If you are working locally, ignore this cell.

In [ ]:
!git clone https://github.com/Fjoelsak/AIBIP.git
!cp AIBIP/06-Generative_Image_Models/Colorization.py ./

**Optional — recommended for faster startup:** If the CIFAR-10 dataset is already stored in your Google Drive, mount Drive and point the data loader to it. This avoids the slow download from the CIFAR-10 server (~170 MB at limited speed).

To prepare once: download CIFAR-10 locally, then upload the `data/` folder to your Drive root. After that, every Colab session can use it instantly.

In [ ]:
# Mount Google Drive and set DATA_ROOT to the folder containing the CIFAR-10 data.
# Skip this cell if you want to download CIFAR-10 directly.

from google.colab import drive
drive.mount("/content/drive")

DATA_ROOT = "/content/drive/MyDrive/data"  # adjust if your folder is named differently

## The Lab Color Space

RGB encodes color as a mixture of red, green, and blue — there is no clean separation between luminance and chrominance. The **CIE Lab** color space decouples them:

- **L**: Lightness (0 = black, 100 = white) — equivalent to a perceptual grayscale
- **a**: Green–red axis (negative = green, positive = red)
- **b**: Blue–yellow axis (negative = blue, positive = yellow)

For colorization this is ideal: the network receives L as input and must predict a and b. We normalize L to [0, 1] and ab to [-1, 1] to match the network's output activations (Sigmoid / Tanh respectively).

## Data Preparation

We download CIFAR-10 and convert all images from RGB to Lab **once at startup**. The `rgb2lab` conversion via scikit-image is slow — running it in `__getitem__` would repeat it for every sample at every epoch (50,000 × 20 = 1M calls). Pre-converting and caching the tensors in RAM eliminates this bottleneck entirely.

The DataLoader then returns `(L, ab)` pairs directly from RAM — no labels are used during training.

In [ ]:
import torch
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms
from skimage.color import rgb2lab

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# Use Drive path if mounted above, otherwise download to local ./data
try:
    DATA_ROOT
except NameError:
    DATA_ROOT = "./data"

print(f"Data root: {DATA_ROOT}")


def precompute_lab(cifar_split: bool) -> tuple[torch.Tensor, torch.Tensor]:
    """Convert a full CIFAR-10 split to cached L and ab tensors."""
    raw = datasets.CIFAR10(root=DATA_ROOT, train=cifar_split, download=True,
                           transform=transforms.ToTensor())
    n = len(raw)
    L_cache  = torch.zeros(n, 1, 32, 32, dtype=torch.float32)
    ab_cache = torch.zeros(n, 2, 32, 32, dtype=torch.float32)

    for i, (img, _) in enumerate(raw):
        img_np = img.permute(1, 2, 0).numpy()
        lab    = rgb2lab(img_np).astype(np.float32)
        L_cache[i, 0]  = torch.from_numpy(lab[:, :, 0] / 100.0)
        ab_cache[i, 0] = torch.from_numpy(lab[:, :, 1] / 128.0)
        ab_cache[i, 1] = torch.from_numpy(lab[:, :, 2] / 128.0)

    return L_cache, ab_cache


print("Pre-converting training set ...")
L_train, ab_train = precompute_lab(True)
print("Pre-converting test set ...")
L_test,  ab_test  = precompute_lab(False)

train_dataset = TensorDataset(L_train, ab_train)
test_dataset  = TensorDataset(L_test,  ab_test)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True,  num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False, num_workers=0)

print(f"Training samples: {len(train_dataset)}, Test samples: {len(test_dataset)}")

### Data Preview

Before training we visualize a few examples: the grayscale L input the network will see, and the ground-truth color image it should reconstruct.

In [ ]:
import matplotlib.pyplot as plt
from skimage.color import lab2rgb

fig, axes = plt.subplots(2, 8, figsize=(14, 4))

for i in range(8):
    L, ab = L_train[i], ab_train[i]

    axes[0, i].imshow(L.squeeze().numpy(), cmap="gray", vmin=0, vmax=1)
    axes[0, i].axis("off")

    lab_img = np.zeros((32, 32, 3), dtype=np.float32)
    lab_img[:, :, 0]  = L.squeeze().numpy() * 100.0
    lab_img[:, :, 1:] = ab.permute(1, 2, 0).numpy() * 128.0
    axes[1, i].imshow(np.clip(lab2rgb(lab_img), 0, 1))
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("L input",      fontsize=9)
axes[1, 0].set_ylabel("Ground truth", fontsize=9)
plt.tight_layout()
plt.show()

## Model Architecture

We use the same encoder-decoder pattern as the autoencoder:

- **Encoder**: Three convolutional blocks (Conv → BatchNorm → ReLU, stride 2) compress `(1, 32, 32)` → `(256, 4, 4)`
- **Decoder**: Three transposed convolutional blocks upsample back to `(2, 32, 32)`, followed by Tanh

The output has **2 channels** (a and b) instead of 1, and uses **Tanh** (range [-1, 1]) instead of Sigmoid.

In [ ]:
from Colorization import ColorizationNet

model = ColorizationNet(base_channels=64).to(device)

_L = torch.zeros(4, 1, 32, 32).to(device)
print("ab output shape:", model(_L).shape)  # expected: (4, 2, 32, 32)

## Training

The loss is MSE between predicted and ground-truth ab channels. No class labels are used — the color image itself provides the supervision signal.

Training for 20 epochs on CIFAR-10 takes approximately 8–12 minutes on a Colab GPU.

In [ ]:
import torch.optim as optim
import torch.nn.functional as F

optimizer = optim.Adam(model.parameters(), lr=1e-3)
epochs    = 20

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for L, ab in train_loader:
        L  = L.to(device, non_blocking=True)
        ab = ab.to(device, non_blocking=True)

        optimizer.zero_grad()
        ab_hat = model(L)
        loss   = F.mse_loss(ab_hat, ab)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1:2d}, Loss: {total_loss / len(train_loader):.6f}")

In [ ]:
model.save_model()

If you do not want to train, you can load the pre-trained colorization model (base_channels=64, 20 epochs, full CIFAR-10 training set).

In [ ]:
import torch
from Colorization import ColorizationNet

device = "cuda" if torch.cuda.is_available() else "cpu"

model = ColorizationNet(base_channels=64).to(device)
model.load_model(path="AIBIP/06-Generative_Image_Models/models/colorization_cifar10.pth", device=device)

## Results: Colorization Quality

We evaluate qualitatively by comparing three rows:
1. **L input** — what the network sees (grayscale)
2. **Prediction** — the network's colorized output
3. **Ground truth** — the original color image

Note that predictions are often **desaturated** (grayish or brownish). This is a known artifact of MSE loss: when the model is uncertain about the correct color, it hedges toward the mean of plausible colors — which is a muted, unsaturated hue. This is not a bug but a fundamental limitation of pixel-wise regression, and one motivation for perceptual losses and generative approaches.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from skimage.color import lab2rgb

model.eval()

L_batch, ab_batch = next(iter(test_loader))
L_batch  = L_batch[:10].to(device)
ab_batch = ab_batch[:10]

with torch.no_grad():
    ab_pred = model(L_batch).cpu()

L_batch = L_batch.cpu()


def to_rgb(L, ab):
    """Reconstruct an RGB image from L (B,1,32,32) and ab (B,2,32,32) tensors."""
    lab = np.zeros((32, 32, 3), dtype=np.float32)
    lab[:, :, 0]  = L.squeeze().numpy() * 100.0
    lab[:, :, 1:] = ab.permute(1, 2, 0).numpy() * 128.0
    return np.clip(lab2rgb(lab), 0, 1)


fig, axes = plt.subplots(3, 10, figsize=(15, 5))
row_labels = ["L input", "Prediction", "Ground truth"]

for i in range(10):
    axes[0, i].imshow(L_batch[i].squeeze().numpy(), cmap="gray", vmin=0, vmax=1)
    axes[1, i].imshow(to_rgb(L_batch[i], ab_pred[i]))
    axes[2, i].imshow(to_rgb(L_batch[i], ab_batch[i]))
    for row in range(3):
        axes[row, i].axis("off")

for row, label in enumerate(row_labels):
    axes[row, 0].set_ylabel(label, fontsize=9)

plt.tight_layout()
plt.show()

## Why MSE Produces Dull Colors

The desaturation effect above is not accidental — it is a direct consequence of the loss function.

For a pixel whose true color could plausibly be red *or* green (e.g. a car that could be either color), the MSE-optimal prediction is the **average** of red and green — which is brown/grey. The network minimizes expected squared error by hedging, not by committing.

We can visualize this by looking at the distribution of predicted vs. ground-truth ab values:

In [ ]:
all_ab_pred = []
all_ab_true = []

model.eval()
with torch.no_grad():
    for L, ab in test_loader:
        ab_hat = model(L.to(device)).cpu()
        all_ab_pred.append(ab_hat.view(-1, 2).numpy())
        all_ab_true.append(ab.view(-1, 2).numpy())

ab_pred_np = np.concatenate(all_ab_pred, axis=0)
ab_true_np = np.concatenate(all_ab_true, axis=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, data, title in zip(
    axes,
    [ab_true_np, ab_pred_np],
    ["Ground truth ab distribution", "Predicted ab distribution"]
):
    ax.hist2d(data[:, 0], data[:, 1], bins=80, range=[[-1, 1], [-1, 1]], cmap="inferno")
    ax.set_xlabel("a channel")
    ax.set_ylabel("b channel")
    ax.set_title(title)

plt.tight_layout()
plt.show()

print(f"Ground truth — std(a): {ab_true_np[:,0].std():.3f}, std(b): {ab_true_np[:,1].std():.3f}")
print(f"Prediction   — std(a): {ab_pred_np[:,0].std():.3f}, std(b): {ab_pred_np[:,1].std():.3f}")

## Discussion

The 2D histogram shows that the ground-truth ab distribution is **spread across the full color gamut**, while the predicted distribution **collapses toward the center** (near-gray). The standard deviation of predictions is significantly smaller than ground truth.

This is the core motivation for going beyond MSE in image synthesis:

| Approach | Idea | Result |
|---|---|---|
| MSE loss | Minimize average pixel error | Correct on average, but desaturated |
| Perceptual loss | Match feature activations (VGG) | Sharper, more saturated |
| Conditional GAN | Discriminator penalizes implausible colors | Vivid, but can hallucinate |
| Diffusion model | Learn full distribution, sample at inference | State of the art |

Despite its limitations, MSE-based colorization successfully demonstrates **self-supervised representation learning**: the network learns meaningful visual features from color images without any human-provided labels.